# DemoEngine: Python y conexión con Flet paso a paso

[00_indice](00_indice.ipynb) · [01_python_y_flet](01_python_y_flet.ipynb) · [02_modelos](02_modelos.ipynb) · [03_controles](03_controles.ipynb) · [04_graficas](04_graficas.ipynb) · [05_eventos](05_eventos.ipynb) · [06_referencia](06_referencia.ipynb) · [07_validacion_e_integracion](07_validacion_e_integracion.ipynb) · [08_conectar_algoritmo](08_conectar_algoritmo.ipynb) · [09_demo_engine_paso_a_paso](09_demo_engine_paso_a_paso.ipynb)

**AM / Evolución Lab · Flet 1.0.4 · Python 3**

## Qué aprenderás
Este capítulo explica **cada apartado de `src/models/demo_engine.py`**, incluyendo sintaxis que suele quedar escondida en expresiones compactas. Léelo en orden y ejecuta las celdas: cada una muestra resultados y comprueba lo explicado con `assert`.

El motor no importa Flet: produce datos que después la interfaz dibuja. Aquí aprenderás Python primero y, al final, cómo se conecta con los controles de Flet. No necesitas abrir una ventana para ejecutar estos ejemplos.

### Mapa del archivo
| Apartado | Propósito |
|---|---|
| Docstring inicial | describe el alcance: datos ilustrativos, sin genética |
| Importaciones | biblioteca estándar y contratos locales |
| initial_population | crea individuos aleatorios dentro de sus dominios |
| DemoEngine | agrupa el comportamiento de un motor reemplazable |
| run | crea la secuencia completa de generaciones |
| axes / target | muestrea el dominio y escoge la mejor posición de referencia |
| progress / population | interpola posiciones para representar cambios |
| scores / Generation | calcula evaluaciones y empaqueta cada generación |
| return result | entrega la lista al consumidor |

## Preparación del kernel
Selecciona el entorno AM. La celda busca el proyecto desde la carpeta actual del kernel. Si estás en otra ubicación, reemplaza esa búsqueda por `project = Path(r'D:\Storage\School\CUCEI\Proyectos\AM')`.


In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
project = next((base / 'AM' for base in candidates if (base / 'AM' / 'src').is_dir()), None)
if project is None:
    project = next(base for base in candidates if (base / 'src').is_dir())
sys.path.insert(0, str(project / 'src'))

import itertools
import random
from models.contracts import AlgorithmConfig, Generation, Individual, Objective, validate_generations
from models.objectives import get_objective
from models.demo_engine import DemoEngine, initial_population

print('Proyecto:', project)

## 1. Módulos, importaciones y docstrings
`import itertools` carga herramientas para iteradores; usamos `product`. `import random` carga generadores pseudoaleatorios; usamos `Random` y `uniform`. Ambos pertenecen a la biblioteca estándar de Python: no requieren `pip install`.

`from models.contracts import ...` trae nombres de nuestro archivo contracts.py. models es un paquete, contracts un módulo y AlgorithmConfig/Generation/Objective son clases. Individual es un alias de tipo, no una clase nueva.

Un texto entre triples comillas al principio de módulo, clase o función es un **docstring**. Explica el objeto y puede consultarse con `help(...)` o `objeto.__doc__`. Un comentario `#` sirve para quien lee el archivo y no se convierte en docstring.

## 2. Lista, tupla, vector y población
Una **lista** `[1,2]` puede modificarse; una **tupla** `(1,2)` no permite sustituir sus elementos. El motor representa cada individuo como tupla y una población aplicada como tupla de individuos.

`Individual = tuple[float, ...]` es una anotación: una tupla de números reales de longitud variable. `...` en este contexto significa que puede haber varios floats; no rellena coordenadas. En 1D escribe `(2.0,)`: la coma crea la tupla; `(2.0)` es sólo un número entre paréntesis.

`list[Individual]` significa lista de individuos. `tuple[Individual, ...]` significa población inmutable. Las anotaciones ayudan a leer el código pero no comprueban datos automáticamente.


In [ ]:
individual_1d = (2.0,)
individual_2d = (2.0, -1.0)
population_example = (individual_2d, (0.0, 3.0))
print(type((2.0)).__name__, type(individual_1d).__name__)
print('Población:', population_example)
print('Primer individuo:', population_example[0])
print('Su primera coordenada:', population_example[0][0])
assert len(population_example) == 2
assert population_example[0][1] == -1.0

## 3. Parámetros, argumentos y atributos
`def initial_population(size: int, objective: Objective) -> list[Individual]:`

- `def`: define una función.
- `size` y `objective`: parámetros, nombres que recibirán valores.
- `: int` y `: Objective`: anotan los tipos esperados.
- `-> list[Individual]`: anota el resultado.
- `:` final y sangría: delimitan el cuerpo.
- `initial_population(3, objective)`: llamada con argumentos posicionales.

`objective.bounds` accede al atributo bounds del objeto. No es un método porque no tiene `()`. `objective.evaluate(vector)` obtiene la función guardada y la ejecuta con ese vector.

### Objective: todos sus campos
| Campo | Contenido | Uso |
|---|---|---|
| name | texto legible | nombre en interfaz |
| expression | fórmula como texto | descripción visual |
| labels | tupla de etiquetas | nombres x,y del editor y ejes |
| bounds | tupla de pares inferior/superior | generar y validar coordenadas |
| evaluate | función que recibe Individual | calcular puntuación |
| dimensions | propiedad calculada con len(labels) | saber cuántas coordenadas hay |

`@dataclass(frozen=True)` en contracts.py genera el constructor y evita reasignar campos. `@property` permite usar `objective.dimensions` en lugar de `objective.dimensions()`. frozen no vuelve inmutable cualquier objeto anidado; en nuestros contratos se usan tuplas para los datos vectoriales.


In [ ]:
objective = get_objective('sphere', 2)
print(objective.name, objective.labels, objective.bounds)
print('Dimensiones:', objective.dimensions)
print('Evaluación de (2,-1):', objective.evaluate((2.0, -1.0)))
assert objective.evaluate((2.0, -1.0)) == 5.0

## 4. Empaquetado y desempaquetado de valores
**Empaquetar** agrupa valores: `point = x, y` produce una tupla aunque no escribas paréntesis. **Desempaquetar** los distribuye: `x, y = point` asigna el primero a x y el segundo a y.

En `for lower, upper in objective.bounds`, cada vuelta obtiene un par y lo desempaqueta. Equivale a obtener bound y después lower=bound[0], upper=bound[1]. Los dos nombres no representan dos bucles: un único bucle recorre pares.

Los niveles importan: `(xmin,xmax), (ymin,ymax) = bounds` desempaqueta primero dos dominios y después cada par. Sin estrella, la cantidad de valores debe coincidir; si no, se produce ValueError.


In [ ]:
x_value, y_value = 2.0, -1.0
packed = x_value, y_value
x_value, y_value = packed
print('Empaquetado:', packed)

for lower, upper in objective.bounds:
    print('Dominio:', lower, upper)

(xmin, xmax), (ymin, ymax) = objective.bounds
assert (xmin, xmax) == objective.bounds[0]
first, *middle, last = [10, 20, 30, 40]
print('Primero, resto y último:', first, middle, last)
assert middle == [20, 30]

## 5. Multiplicar listas y tuplas: repetición, no escalado numérico
`[1,2] * 3` repite la secuencia tres veces. `(1,2) * 3` hace lo mismo conservando tipo tupla. No produce `[3,6]`. Para multiplicar los números, usa `[v*3 for v in values]`.

En objectives.py, `(domains[key],) * dimensions` repite el par de dominio. La coma crea una tupla **que contiene un par**. Repetirla conserva la estructura `((mín,máx),(mín,máx))`. Es seguro compartir esos pares porque son tuplas de números inmutables.

En PopulationEditor, `(None,) * dimensions` crea un límite opcional por coordenada. El None de cada posición indica que no hay cotas.

### Peligro con listas anidadas
`[[0,0]] * 3` repite referencias a la **misma lista interior**. Cambiar una fila cambia las tres. Usa `[[0,0] for _ in range(3)]` para crear tres listas independientes. Una tupla también puede contener una lista mutable; la inmutabilidad exterior no evita modificar esa lista.


In [ ]:
print([1, 2] * 3)
print((1, 2) * 3)
print([value * 3 for value in [1, 2]])
print(((-5, 5),) * 2)
assert ((-5, 5),) * 2 == ((-5, 5), (-5, 5))

shared_rows = [[0, 0]] * 3
shared_rows[0][0] = 9
assert shared_rows == [[9, 0], [9, 0], [9, 0]]
assert shared_rows[0] is shared_rows[1]

independent_rows = [[0, 0] for _ in range(3)]
independent_rows[0][0] = 9
assert independent_rows == [[9, 0], [0, 0], [0, 0]]
print('Filas compartidas:', shared_rows)
print('Filas independientes:', independent_rows)

## 6. Comprensión de listas: un for dentro de corchetes
Sintaxis: `[expresión for variable in iterable]`. Léela como: **para cada variable en iterable, calcula expresión y guarda el resultado en una lista**. El resultado se crea inmediatamente.

`[i*i for i in range(4)]` equivale a crear lista vacía, recorrer range y hacer append(i*i). La expresión aparece primero al escribir, pero al leer conviene empezar por el for.

También existe filtro: `[i*i for i in range(4) if i%2 == 0]`. Ese if decide qué elementos se incluyen. No es igual al if/else que elige entre dos valores, explicado después.


In [ ]:
compact = [i*i for i in range(4)]
expanded = []
for i in range(4):
    expanded.append(i*i)
assert compact == expanded == [0, 1, 4, 9]
print(compact)

only_even = [i*i for i in range(4) if i % 2 == 0]
assert only_even == [0, 4]

## 7. Generadores y tuple(...)
`(expresión for variable in iterable)` crea una **expresión generadora**: entrega valores cuando se le piden. No crea una tupla automáticamente. `tuple(generador)` consume esos valores y crea la tupla.

`tuple(function(v) for v in values)` permite omitir los paréntesis adicionales del generador cuando es el único argumento. Equivale a `tuple((function(v) for v in values))`.

Un generador se agota: recorrerlo una segunda vez no vuelve al principio. Una lista o tupla sí puede recorrerse varias veces. En este motor los generadores son cortos y se materializan inmediatamente con tuple, por lo que las generaciones finales contienen datos reutilizables.


In [ ]:
generator = (i*i for i in range(4))
print(type(generator).__name__)
materialized = tuple(generator)
assert materialized == (0, 1, 4, 9)
assert tuple(generator) == ()  # Ya se consumió.
print('Tupla resultante:', materialized)

## 8. initial_population: validación y aleatoriedad
`if not 2 <= size <= 200:` es una comparación encadenada: acepta size entre 2 y 200 inclusive. `not` invierte la condición. `raise ValueError(...)` interrumpe la función cuando el tamaño es inválido. La interfaz ya convierte el campo a entero; esta función no comprueba todos los tipos posibles por sí sola.

`rng = random.Random(42)` crea un objeto generador local. 42 es la semilla: repetir esta llamada con igual tamaño y dominio produce la misma muestra dentro del mismo entorno. No es un porcentaje ni una coordenada. Cada llamada a uniform consume un nuevo valor del generador.

`rng.uniform(lower,upper)` devuelve un float entre los límites. El objeto rng tiene estado interno; no reinicies la semilla para cada coordenada porque repetirías valores innecesariamente.

### Descomponer la línea compacta
```python
return [tuple(rng.uniform(lower, upper) for lower, upper in objective.bounds)
        for _ in range(size)]
```

1. El for exterior repite size veces: una vez por individuo.
2. `_` es un nombre de variable convencional para un valor que no utilizaremos; no es una palabra reservada.
3. El generador interior recorre cada dominio y desempaqueta lower/upper.
4. uniform obtiene una coordenada por dominio.
5. tuple reúne esas coordenadas en un individuo.
6. Los corchetes exteriores reúnen los individuos en una lista.

`range(size)` produce enteros 0 hasta size−1. El límite superior se excluye. Son índices de repetición, no coordenadas.


In [ ]:
def initial_population_expanded(size, objective):
    if not 2 <= size <= 200:
        raise ValueError('La cantidad para generar debe estar entre 2 y 200.')
    rng = random.Random(42)
    population = []
    for individual_index in range(size):
        coordinates = []
        for lower, upper in objective.bounds:
            coordinates.append(rng.uniform(lower, upper))
        individual = tuple(coordinates)
        population.append(individual)
    return population

compact_population = initial_population(3, objective)
expanded_population = initial_population_expanded(3, objective)
assert compact_population == expanded_population
for index, individual in enumerate(compact_population):
    print(index, individual)

## 9. Clase DemoEngine, self y run
`class DemoEngine:` define un tipo que agrupa un método. `engine = DemoEngine()` crea una instancia. No define __init__, por lo que usa la inicialización básica heredada de object. El motor no guarda config como atributo; las variables de run son locales a cada llamada.

`def run(self, config: AlgorithmConfig) -> list[Generation]:` define un método. self es la instancia recibida automáticamente. `engine.run(config)` equivale conceptualmente a `DemoEngine.run(engine,config)`.

La clase cumple el contrato EvolutionEngine porque proporciona run con la entrada y salida esperadas. No es necesario escribir herencia explícita del Protocol. Protocol describe una interfaz estructural; no implementa selección o mutación por ti.

### Todos los campos de config
| Atributo | Papel |
|---|---|
| population | tupla de vectores iniciales |
| objective | objeto con evaluate, dominios y etiquetas |
| mutation_rate | probabilidad [0,1], disponible para tu futuro algoritmo |
| selection | clave torneo/ruleta/elitista, disponible para tu futuro algoritmo |
| generations | iteraciones positivas; se agrega generación 0 |

En DemoEngine, mutation_rate y selection **no se usan para producir cambios**. run presupone configuración válida; la interfaz valida generaciones entre 1 y 500 y población antes de llamarlo. Una llamada manual con generations=0 produciría división entre cero: debes validar antes de usarla.

## 10. axes: comprensión de listas dentro de otra lista
```python
axes = [[lower+(upper-lower)*i/60 for i in range(61)]
        for lower, upper in objective.bounds]
```

El for exterior crea una lista por dimensión. El interior crea 61 muestras en el dominio de esa dimensión. El resultado es una lista **de listas**, no una única lista plana.

`upper-lower` mide amplitud; `i/60` es progreso entre 0 y 1. `/` produce división real; `//` sería división entera hacia abajo. `lower + amplitud*progreso` coloca el punto dentro del dominio. Los paréntesis hacen que la resta se calcule antes de multiplicar. Como range(61) incluye 0 y 60, se incluyen ambos extremos y hay 60 intervalos entre 61 puntos.


In [ ]:
axes_compact = [[lower+(upper-lower)*i/60 for i in range(61)] for lower, upper in objective.bounds]
axes_expanded = []
for lower, upper in objective.bounds:
    axis_points = []
    for i in range(61):
        point = lower + (upper-lower) * i / 60
        axis_points.append(point)
    axes_expanded.append(axis_points)
assert axes_compact == axes_expanded
assert len(axes_compact) == 2
assert all(len(axis) == 61 for axis in axes_compact)
print('Primeros puntos de x:', axes_compact[0][:4])
print('Últimos puntos de x:', axes_compact[0][-4:])

## 11. Anidación frente a dos for en la misma comprensión
`[[... for ...] for ...]` mantiene filas. `[... for ... for ...]` recorre bucles anidados, pero guarda resultados en **una sola lista**. Lee los for de izquierda a derecha como bucles tradicionales.

El motor usa la primera forma para axes porque necesita conservar cada eje separado. La segunda forma sería útil para una lista plana de combinaciones, pero cambiaría la estructura.


In [ ]:
nested = [[x+y for y in [10, 20]] for x in [1, 2]]
flat = [x+y for x in [1, 2] for y in [10, 20]]
assert nested == [[11, 21], [12, 22]]
assert flat == [11, 21, 12, 22]
print('Anidada:', nested)
print('Plana:', flat)

## 12. La estrella: expandir argumentos, no multiplicar
La misma estrella cambia de significado según dónde esté:

| Expresión | Significado |
|---|---|
| values * 2 | repetir secuencia o multiplicar un número |
| function(*values) | desempaquetar iterable en argumentos posicionales |
| function(**options) | desempaquetar diccionario en argumentos nombrados |
| def function(*args) | empaquetar argumentos posicionales recibidos en una tupla |
| def function(**kwargs) | empaquetar argumentos nombrados recibidos en un diccionario |
| first, *rest = values | asignar primer valor y resto como lista |

`itertools.product(*axes)` no multiplica axes. Si axes tiene dos listas, equivale a `itertools.product(axes[0],axes[1])`. El número de argumentos se adapta a las dimensiones.

En theme.py, `field(..., **kwargs)` recibe opciones de Flet y las reenvía con `ft.TextField(..., **kwargs)`. kwargs es un nombre convencional, no una palabra reservada.


In [ ]:
def pack_arguments(*args, **kwargs):
    return args, kwargs

positional = (1, 2)
named = {'label':'x', 'width':120}
args, kwargs = pack_arguments(*positional, **named)
assert args == (1, 2)
assert kwargs == named
print('args:', args, 'kwargs:', kwargs)

## 13. itertools.product y zip: combinaciones frente a parejas
product(xs,ys) produce **todas las combinaciones**: con 2 elementos en x y 3 en y hay 6 vectores. Devuelve un iterador; el motor lo consume mediante min.

zip(xs,ys) empareja por posición y termina cuando se agota la secuencia más corta. No produce todas las combinaciones. Se usa después para unir coordenadas actuales con las coordenadas del objetivo.

En el muestreo de run hay 61 posiciones por dimensión: 61 candidatos en 1D y 61²=3721 en 2D. Para más dimensiones el costo crece como 61 elevado a d; no conviene generalizar este muestreo indiscriminadamente. Las funciones de ejemplo de la app admiten 1D/2D.


In [ ]:
small_axes = [[0, 1], [10, 20, 30]]
combinations = list(itertools.product(*small_axes))
pairs = list(zip(*small_axes))
print('product:', combinations)
print('zip:', pairs)
assert len(combinations) == 6
assert pairs == [(0, 10), (1, 20)]

## 14. min(..., key=objective.evaluate)
```python
target = min(itertools.product(*axes), key=objective.evaluate)
```

min recibe un iterable de candidatos. key es una función que calcula el valor por el que se comparan. Devuelve **el candidato**, no la puntuación mínima. `objective.evaluate` pasa la referencia a la función; escribir `objective.evaluate()` la ejecutaría sin el vector que necesita.

Sin key, Python compara tuplas lexicográficamente: compara primero x y después y. Eso no encuentra necesariamente el menor f(x,y). Si hay empate, min conserva el primer candidato encontrado.

target es el mínimo **entre las muestras**, no una prueba del mínimo global continuo. Se calcula una vez antes del bucle de generaciones, no se vuelve a buscar por cada individuo.


In [ ]:
target = min(itertools.product(*axes_compact), key=objective.evaluate)
assert target == (0.0, 0.0)  # La malla de esfera incluye el origen.
print('Posición objetivo:', target, 'evaluación:', objective.evaluate(target))

candidates_example = [(0.0, 4.0), (1.0, 0.0)]
assert min(candidates_example) == (0.0, 4.0)
assert min(candidates_example, key=objective.evaluate) == (1.0, 0.0)

## 15. Generaciones, progress y operadores
`result = []` crea la lista de salidas. `for index in range(config.generations+1)` incluye índices desde 0 hasta generations. El +1 compensa que range excluye el límite. Si hay 4 iteraciones, el motor devuelve 5 generaciones: 0,1,2,3,4.

`progress = index/config.generations` produce un número entre 0 y 1. Para una coordenada inicial v y destino t, la expresión `v+(t-v)*progress` es una interpolación:

- progress=0: conserva v.
- progress=0.5: posición a mitad de camino.
- progress=1: alcanza t, salvo redondeos propios de floats.

El signo menos calcula desplazamiento, la multiplicación aplica fracción y la suma añade el punto inicial. No son operadores de cruce ni mutación.


In [ ]:
v, t = 4.0, 0.0
trajectory = []
for index in range(4+1):
    progress = index/4
    position = v+(t-v)*progress
    trajectory.append(position)
    print('Generación', index, 'progreso', progress, 'posición', position)
assert trajectory == [4.0, 3.0, 2.0, 1.0, 0.0]

## 16. Expresión condicional if/else
`valor_si_verdadero if condición else valor_si_falso` elige una expresión. En el motor:

```python
population = config.population if index == 0 else ...
```

Si index==0, utiliza la población original sin recalcular coordenadas. En otro caso, calcula los nuevos vectores. Sólo se evalúa la rama elegida. `==` compara; `=` asigna.

Equivale a un if/else tradicional con asignación en cada rama. No es el filtro de una comprensión: siempre devuelve un valor.

## 17. La población interpolada: dos generadores y dos tuple
```python
tuple(tuple(v+(t-v)*progress for v,t in zip(individual,target))
      for individual in config.population)
```

1. El generador exterior recorre cada individuo original.
2. zip(individual,target) empareja cada coordenada actual con su destino.
3. `for v,t` desempaqueta cada pareja.
4. La expresión numérica calcula una coordenada nueva.
5. tuple interior reúne coordenadas de **un individuo**.
6. tuple exterior reúne individuos de **la población**.

En este contrato, ambos vectores tienen la misma dimensión. Como zip corta al más corto, validar dimensiones antes evita perder coordenadas silenciosamente.


In [ ]:
original_population = ((4.0, 2.0), (-2.0, -4.0))
target_example = (0.0, 0.0)
progress_example = 0.5

compact_new = tuple(tuple(v+(t-v)*progress_example for v,t in zip(individual,target_example)) for individual in original_population)

new_individuals = []
for individual in original_population:
    new_coordinates = []
    for v, t in zip(individual, target_example):
        coordinate = v+(t-v)*progress_example
        new_coordinates.append(coordinate)
    new_individuals.append(tuple(new_coordinates))
expanded_new = tuple(new_individuals)
assert compact_new == expanded_new == ((2.0, 1.0), (-1.0, -2.0))
print('Antes:', original_population, 'después:', compact_new)

## 18. Puntuaciones y empaquetado en Generation
```python
tuple(objective.evaluate(individual) for individual in population)
```

Este generador recorre la población nueva y calcula una puntuación por individuo. tuple materializa puntuaciones en el mismo orden que sus vectores.

`Generation(index,population,scores)` construye una instancia de la dataclass Generation. No es una tupla de tres elementos: es un objeto con atributos nombrados. El orden de argumentos posicionales corresponde al de los campos declarados en contracts.py; también puedes escribir `Generation(index=index,population=population,scores=scores)`.

`result.append(generation)` añade una generación a la lista y devuelve None. No hagas `result = result.append(...)`, porque reemplazarías la lista por None. `return result` termina run y entrega todas las generaciones.


In [ ]:
scores_compact = tuple(objective.evaluate(individual) for individual in expanded_new)
scores_list = []
for individual in expanded_new:
    scores_list.append(objective.evaluate(individual))
scores_expanded = tuple(scores_list)
assert scores_compact == scores_expanded == (5.0, 5.0)
snapshot = Generation(index=1, population=expanded_new, scores=scores_compact)
print(snapshot.index, snapshot.population, snapshot.scores)
results_example = []
assert results_example.append(snapshot) is None
assert results_example == [snapshot]

## 19. run completo con bucles tradicionales
La siguiente versión realiza las mismas operaciones en el mismo orden, manteniendo la misma malla y destino. Su propósito es pedagógico; no reemplaza el archivo de la app. Compara cada apartado con la versión compacta.


In [ ]:
def run_expanded(config):
    objective = config.objective

    # A. Una lista de muestras por dimensión.
    axes = []
    for lower, upper in objective.bounds:
        axis = []
        for i in range(61):
            axis.append(lower+(upper-lower)*i/60)
        axes.append(axis)

    # B. Posición con menor evaluación entre todas las combinaciones.
    target = min(itertools.product(*axes), key=objective.evaluate)

    # C. Construir la secuencia de generaciones.
    result = []
    for index in range(config.generations+1):
        progress = index/config.generations
        if index == 0:
            population = config.population
        else:
            new_population = []
            for individual in config.population:
                coordinates = []
                for v, t in zip(individual, target):
                    coordinates.append(v+(t-v)*progress)
                new_population.append(tuple(coordinates))
            population = tuple(new_population)

        # D. Evaluar y guardar población del paso actual.
        scores = []
        for individual in population:
            scores.append(objective.evaluate(individual))
        generation = Generation(index, population, tuple(scores))
        result.append(generation)
    return result

config_example = AlgorithmConfig(tuple(initial_population(3, objective)), objective, 0.05, 'torneo', 4)
compact_result = DemoEngine().run(config_example)
expanded_result = run_expanded(config_example)
assert compact_result == expanded_result
validate_generations(compact_result, config_example)
assert len(compact_result) == 5
for generation in compact_result:
    print(generation.index, generation.population[0], generation.scores[0])

## 20. Código vigente para comparar
El siguiente bloque se copia automáticamente del archivo al generar la documentación. Los ejemplos anteriores lo descomponen; el código de la app no se cambia para crear este capítulo.

```python
"""Motor ilustrativo reemplazable; no implementa selección, cruce ni mutación."""
import itertools
import random
from models.contracts import AlgorithmConfig, Generation, Individual, Objective


def initial_population(size: int, objective: Objective) -> list[Individual]:
    """Genera size vectores reproducibles dentro de los límites de cada coordenada."""
    if not 2 <= size <= 200:
        raise ValueError("La cantidad para generar debe estar entre 2 y 200.")
    rng = random.Random(42)
    return [tuple(rng.uniform(lower, upper) for lower, upper in objective.bounds) for _ in range(size)]


class DemoEngine:
    """Implementación ilustrativa de EvolutionEngine que puedes reemplazar por tu motor."""

    def run(self, config: AlgorithmConfig) -> list[Generation]:
        """Interpola hacia un mínimo muestreado; selección y mutación no actúan."""
        objective = config.objective
        axes = [[lower+(upper-lower)*i/60 for i in range(61)] for lower, upper in objective.bounds]
        target = min(itertools.product(*axes), key=objective.evaluate)
        result = []
        for index in range(config.generations+1):
            progress = index/config.generations
            population = config.population if index == 0 else tuple(tuple(v+(t-v)*progress for v, t in zip(individual, target)) for individual in config.population)
            result.append(Generation(index, population, tuple(objective.evaluate(individual) for individual in population)))
        return result

```

## 21. Cómo llegan estos datos a Flet
| Paso | Apartado | Datos / operación |
|---|---|---|
| 1 | PopulationEditor | fields contiene TextField; field.value es texto ingresado |
| 2 | PopulationEditor.save | float convierte cada entrada, tuple empaqueta coordenadas |
| 3 | Dashboard.settings | lee controles y construye AlgorithmConfig |
| 4 | Dashboard.apply | await asyncio.to_thread(engine.run,config) ejecuta cálculo fuera del hilo de interfaz |
| 5 | validate_generations | comprueba estructura y finitud de salida |
| 6 | Dashboard.snapshots | conserva lista de Generation ya calculada |
| 7 | Dashboard.render | index elige una generación y scores identifica mejor/promedio |
| 8 | Chart.set_image | Image.src recibe bytes PNG de gráfica |
| 9 | page.update | comunica propiedades modificadas al cliente Flet |

### Propiedades de controles relacionadas
`TextField.value` es texto, no el float del cromosoma. `Dropdown.value` es una clave elegida, como 'torneo'. `Slider.value` es número: mutación visual 5 se convierte a 0.05 mediante división por 100. `Button.on_click` y `TextField.on_submit` guardan callbacks; pasar método sin paréntesis lo registra sin ejecutarlo.

`ListView.controls` contiene las filas visibles del editor; son controles, no la población numérica. `Image.src` contiene imagen codificada, no posiciones. `visible` y `disabled` son estados visuales. `page.update()` transmite modificaciones del árbol de controles; DemoEngine.run no debe llamarlo ni importar Flet.

`async def` crea coroutine; `await` espera sin bloquear el bucle cuando la operación lo permite. `asyncio.to_thread` recibe **referencia de función** y argumentos: no escribas `to_thread(engine.run(config))`, porque ejecutarías el cálculo antes de pasarlo al hilo. La reproducción usa asyncio.sleep para recorrer resultados ya calculados, no ejecuta el motor por cada frame.

## 22. Dónde introducir tu algoritmo real
Conserva AlgorithmConfig y Generation. Reemplaza DemoEngine por una clase con run(config) que aplique selección, cruce y mutación. En cada iteración guarda la población resultante y sus evaluaciones. initial_population puede seguir sirviendo para generar entradas.

En esta demostración cada posición se calcula **desde la población inicial** y progress; no desde la población anterior. Es interpolación ilustrativa. Un algoritmo genético normalmente produce la siguiente población mediante operadores sobre individuos actuales. No confundir el mínimo muestreado target con una selección genética.

## 23. Guía rápida para leer expresiones compactas
1. Identifica estructura exterior: lista, tuple o llamada a función.
2. Busca el for exterior y qué representa cada elemento.
3. Si hay otro for, identifica si es anidado en otra lista o si aplana el resultado.
4. Descompón el cuerpo en variables locales: coordenada, individuo, población, puntuación.
5. Expande la expresión con lista vacía y append; verifica con assert que ambas versiones coinciden.
6. Distingue referencia de función de ejecución, repetición de secuencia de expansión de argumentos y dato numérico de control Flet.

Estos son recursos de Python, no sintaxis especial de Flet. Los mismos patrones se pueden usar sin una interfaz gráfica.
